# apply & lambda (cours)

Exemples à **exécuter** (contexte : des produits). Les exercices (des employés) sont tout en bas.

### Un petit df de produits

In [36]:
import pandas as pd
prod = pd.DataFrame({"nom":["Clavier","Souris","Écran","Cahier"], "prix":[89.9, 29.9, 219.0, 4.5]})
print(prod)

       nom   prix
0  Clavier   89.9
1   Souris   29.9
2    Écran  219.0
3   Cahier    4.5


### apply + lambda : transformer chaque valeur

In [37]:
prod["prix_ttc"] = prod["prix"].apply(lambda p: round(p * 1.2, 2))
print(prod)

       nom   prix  prix_ttc
0  Clavier   89.9    107.88
1   Souris   29.9     35.88
2    Écran  219.0    262.80
3   Cahier    4.5      5.40


### def + apply : catégoriser avec plusieurs conditions

In [38]:
def gamme(prix):
    if prix > 200:
        return "cher"
    elif prix > 50:
        return "moyen"
    else:
        return "abordable"

prod["gamme"] = prod["prix"].apply(gamme)
print(prod)

       nom   prix  prix_ttc      gamme
0  Clavier   89.9    107.88      moyen
1   Souris   29.9     35.88  abordable
2    Écran  219.0    262.80       cher
3   Cahier    4.5      5.40  abordable


### apply axis=1 : croiser plusieurs colonnes (ligne par ligne)

In [39]:
def etiquette(ligne):
    return f'{ligne["nom"]} ({ligne["gamme"]})'
prod["label"] = prod.apply(etiquette, axis=1)
print(prod[["nom","gamme","label"]])

       nom      gamme               label
0  Clavier      moyen     Clavier (moyen)
1   Souris  abordable  Souris (abordable)
2    Écran       cher        Écran (cher)
3   Cahier  abordable  Cahier (abordable)


### Piège : apply lent vs vectorisé

In [40]:
# ❌ apply pour une simple multiplication (lent) :
# prod["ttc"] = prod["prix"].apply(lambda p: p * 1.2)
# ✅ vectorisé (pandas fait tout d'un coup) :
prod["ttc"] = prod["prix"] * 1.2
print(prod[["prix","ttc"]])

    prix     ttc
0   89.9  107.88
1   29.9   35.88
2  219.0  262.80
3    4.5    5.40


# Exercices

Données : `data/employes.csv` (`nom`, `service`, `salaire`, `anciennete` en années). **À toi de choisir tes outils.**

1. Crée `tranche_salaire` : `"bas"` si salaire < 35000, `"moyen"` entre 35000 et 45000 **inclus**, `"élevé"` au-dessus. (règle à plusieurs conditions)
2. Crée `salaire_annuel_charge` = salaire × 1,42. **Quelle est la façon la plus efficace de faire, et pourquoi ?** Justifie en commentaire.
3. Crée `prime` calculée **ligne par ligne** : service `"Data"` **et** au moins **5 ans** d'ancienneté → **12 %** du salaire ; sinon → **6 %**. (croise deux colonnes)
4. Combien d'employés touchent la prime à 12 % ? (déduis-le du résultat de la question 3)
5. Réflexion (pas de code) : quand utiliser `apply`, et quand vaut-il mieux **ne pas** l'utiliser ?

In [41]:
# --- CELLULE FOURNIE : données chargées pour toi ---
import pandas as pd
employes = pd.read_csv("data/employes.csv")
print(employes.shape)
employes.head()

(20, 4)


,nom,service,salaire,anciennete
0,Alice,Marketing,38000,5
1,Bob,RH,28000,8
2,Chloé,Data,44000,10
3,David,Marketing,34000,3
4,Emma,Data,34000,9


In [42]:
## 1
def tranche_salaire (salaire):
    if salaire < 35000:
        return "bas"
    if salaire >= 35000 and salaire <= 45000:
        return "moyen"
    else:
        return "élevé"
    
employes["Tranche salaire"] = employes["salaire"].apply(tranche_salaire)  
employes

,nom,service,salaire,anciennete,Tranche salaire
0,Alice,Marketing,38000,5,moyen
1,Bob,RH,28000,8,bas
2,Chloé,Data,44000,10,moyen
3,David,Marketing,34000,3,bas
4,Emma,Data,34000,9,bas
5,Farid,RH,52000,6,élevé
6,Gaël,Finance,47000,2,élevé
7,Hana,Finance,34000,1,bas
8,Ines,Marketing,44000,0,moyen
9,Jules,RH,38000,7,moyen


In [43]:
## 2
# employes["salaire_annuel_charge"] = employes["salaire"] *1.42

employes["salaire_annuel_charge"] = employes["salaire"].apply(lambda x:x*1.42)
employes

# La version 1 est plus courte à écrire et à exécuter si on avait des millions de lignes par exemple
# Donc pour moi la meilleur façon de faire c'est la première en faisant uniquement  employes["salaire"] * 1.42

,nom,service,salaire,anciennete,Tranche salaire,salaire_annuel_charge
0,Alice,Marketing,38000,5,moyen,53960.0
1,Bob,RH,28000,8,bas,39760.0
2,Chloé,Data,44000,10,moyen,62480.0
3,David,Marketing,34000,3,bas,48280.0
4,Emma,Data,34000,9,bas,48280.0
5,Farid,RH,52000,6,élevé,73840.0
6,Gaël,Finance,47000,2,élevé,66740.0
7,Hana,Finance,34000,1,bas,48280.0
8,Ines,Marketing,44000,0,moyen,62480.0
9,Jules,RH,38000,7,moyen,53960.0


In [47]:
## 3
def prime_salaire (employes):
    if employes["service"] == "Data" and employes["anciennete"] >= 5:
        return employes["salaire"] * 0.12
        
    else: 
        return employes["salaire"] * 0.06
    
employes["prime"] = employes.apply(prime_salaire, axis=1)
employes


,nom,service,salaire,anciennete,Tranche salaire,salaire_annuel_charge,prime
0,Alice,Marketing,38000,5,moyen,53960.0,2280.0
1,Bob,RH,28000,8,bas,39760.0,1680.0
2,Chloé,Data,44000,10,moyen,62480.0,5280.0
3,David,Marketing,34000,3,bas,48280.0,2040.0
4,Emma,Data,34000,9,bas,48280.0,4080.0
5,Farid,RH,52000,6,élevé,73840.0,3120.0
6,Gaël,Finance,47000,2,élevé,66740.0,2820.0
7,Hana,Finance,34000,1,bas,48280.0,2040.0
8,Ines,Marketing,44000,0,moyen,62480.0,2640.0
9,Jules,RH,38000,7,moyen,53960.0,2280.0


In [45]:
## 4
employes[employes["service"]=="Data"].sort_values(by="prime", ascending=False)

# Il semblerait que 3 employes (Chloé, Léa et Emma) aient touché la prime uniquement 


,nom,service,salaire,anciennete,Tranche salaire,salaire_annuel_charge,prime
2,Chloé,Data,44000,10,moyen,62480.0,5280.0
11,Lea,Data,38000,10,moyen,53960.0,4560.0
4,Emma,Data,34000,9,bas,48280.0,4080.0
17,Sami,Data,34000,2,bas,48280.0,2040.0
19,Uma,Data,34000,1,bas,48280.0,2040.0


In [46]:
## 5 (réflexion en commentaire)
# apply : pour appliquer des fonction sur plusieurs colonnes (avec axis) ou de petits datasets 
# vectorisé : quand on a un énorme dataset ou qu'on a un calcul "tout con" comme x * 1.5 (avec x = une colonne du df)